# Public source intake — CPU runtime

This downloads **6.890 GB** with the default `outdoor` profile: two original
AuDroK outdoor archives, the small dual-drone archive and catalog, and one
RealMAN background microphone file used to verify selective extraction.
No GPU or model training is involved. Select a **CPU runtime**.

Confirm enough available Drive allocation before running. `STORAGE_BUDGET_GIB`
is your declared project allocation; it does not measure the account's quota.
The `pilot` profile downloads only 206 MB. Both profiles preserve native sources.

This is acquisition for inspection. Repeated microphones are not independent
capture time, publisher labels still need review, and this intake does not pass
the full corpus diversity/quality gate. Keep one active runtime per intake folder.


In [ ]:
PROJECT_DRIVE_PATH = '/content/drive/MyDrive/drone-detection-v2/intake-v1'
INTAKE_PROFILE = 'outdoor'  # Or 'pilot' for the 206 MB verified sample.
STORAGE_BUDGET_GIB = 20.0  # Set to the available allocation you have checked.
REPO_URL = 'https://github.com/StoneDerek/drone-detection.git'
FIRST_REF = 'main'


In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
MOUNT = Path('/content/drive')
if not os.path.ismount(MOUNT):
    raise RuntimeError('Drive is not mounted; refusing an ephemeral intake folder')
PROJECT = Path(PROJECT_DRIVE_PATH).resolve()
if PROJECT == MOUNT.resolve() or not PROJECT.is_relative_to(MOUNT.resolve()):
    raise ValueError('Choose a project folder inside mounted Drive')
PROJECT.mkdir(parents=True, exist_ok=True)


In [ ]:
"""Standard-library-only bootstrap, embedded verbatim in the Colab notebook."""

import hashlib
import json
import os
import re
import subprocess
import uuid
from pathlib import Path


def git_output(checkout, *arguments):
    return subprocess.check_output(['git', '-C', str(checkout), *arguments], text=True).strip()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def pin_checkout(project, checkout, repo_url, first_ref='main'):
    """Resolve once, then reuse that exact revision after runtime replacement.

    One active runtime per project. Refuse dirty or unrelated local checkouts.
    The saved lock hash is also checked; a pin is never silently refreshed.
    """
    project, checkout = Path(project), Path(checkout)
    project.mkdir(parents=True, exist_ok=True)
    pin_path = project / 'bootstrap.json'
    pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if pin is not None and (pin.get('schema_version') != 1
                            or pin.get('repo_url') != repo_url
                            or not re.fullmatch(r'[0-9a-f]{40}', pin.get('revision', ''))
                            or not re.fullmatch(r'[0-9a-f]{64}', pin.get('runtime_lock_sha256', ''))):
        raise ValueError('Invalid/different project pin; choose a new project folder deliberately')
    if checkout.exists() and any(checkout.iterdir()):
        if not (checkout / '.git').exists():
            raise ValueError('Checkout folder contains unrelated files')
        if git_output(checkout, 'remote', 'get-url', 'origin') != repo_url:
            raise ValueError('Checkout has a different repository origin')
        if git_output(checkout, 'status', '--porcelain'):
            raise ValueError('Local checkout changed; preserve those changes before continuing')
    else:
        checkout.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'init', '--quiet', str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'remote', 'add', 'origin', repo_url], check=True)
    target = pin['revision'] if pin else first_ref
    subprocess.run(['git', '-C', str(checkout), 'fetch', '--quiet', '--depth', '1', 'origin', target], check=True)
    revision = git_output(checkout, 'rev-parse', 'FETCH_HEAD')
    if pin and revision != pin['revision']:
        raise ValueError('Fetched revision differs from saved project pin')
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--quiet', '--detach', revision], check=True)
    candidate = {'schema_version': 1, 'repo_url': repo_url, 'revision': revision,
                 'runtime_lock_sha256': file_hash(checkout / 'notebooks/runtime/uv.lock')}
    if pin is not None and pin != candidate:
        raise ValueError('Pinned environment lock changed')
    if pin is None:
        payload = json.dumps(candidate, indent=2, sort_keys=True).encode()
        temporary = pin_path.with_name('bootstrap.partial-' + uuid.uuid4().hex)
        temporary.write_bytes(payload)
        if pin_path.exists() and json.loads(pin_path.read_text()) != candidate:
            raise ValueError('Another bootstrap wrote a different pin; use one active runtime')
        os.replace(temporary, pin_path)
        if pin_path.read_bytes() != payload:
            raise ValueError('Project pin read-back failed')
    return candidate


REPO = Path('/content/drone-source-intake')
PIN = pin_checkout(PROJECT, REPO, REPO_URL, FIRST_REF)
print(json.dumps(PIN, indent=2))


This intake has its own code pin, separate from the completed GPU recovery
probe. Replacement runtimes reuse its exact commit. No packages are installed;
the downloader uses Python's standard library.

The known pilot files must match pinned SHA-256 values. The two large outdoor
archives have no publisher checksum; their first full transfer checks the
advertised byte count, hashes the bytes, verifies storage read-back and freezes
the observed hash in a receipt. This is not publisher checksum verification.

AuDroK's server ignores HTTP ranges. If a runtime dies during an archive, that
archive restarts; completed archives are rehashed and reused. Partial files are
not accepted as complete. A rerun rejects changed/corrupt sources and receipts.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(REPO / 'scripts/fetch_source_intake.py'),
                '--manifest', str(REPO / 'data/source-intake-v2.json'),
                '--profile', INTAKE_PROFILE,
                '--output', str(PROJECT / 'source-archives'),
                '--budget-gib', str(STORAGE_BUDGET_GIB)], cwd=REPO, check=True)
print('Source directory:', PROJECT / 'source-archives')


Run the flush cell, replace the runtime, and repeat this notebook to check
that completed sources persist and are reused. Mounted read-back alone does not
independently prove cloud sync. Keep source archives and receipts together.

Next comes archive/member inspection, microphone/recording parentage, native
precision and interval validation, followed by a frozen data manifest. No
train/test assignment or new detector result is produced by this notebook.


In [ ]:
drive.flush_and_unmount()
print('Drive flushed and unmounted. Completed sources can be verified on the next runtime.')
